# Introduction
Welcome to the first practical for Graph Representation Learning (MT22). In this practical, we will be covering content from lectures 4 and 5.

We will be using [PyTorch](https://pytorch.org/docs/stable/index.html) to implement TransE from scratch, building it up piece by piece.

The main goal of the practical is creating a working implementation of TransE. There are also two optional parts of the practical: *filtered negative sampling* and *RotatE*.

The notebook is divided into sections, each of which comes with complete or partially completed code. Before each snippet of code there will be a description of what we are about to implement. The sections of code you need to complete are marked as **Tasks**. The majority of the length of this practical comes from code already written for you, so don't panic at the apparent length. There are only 8 tasks for you to complete.

Please ensure that you operate within the framework given in the notebook and bring any questions you may have to the practical demonstrators. We suggest that you **DO NOT** edit code that is a part of the framework, since this will make it more difficult for demonstrators to assist if your code is broken.

Since we are working in a Jupyter Notebook, the code is very interactive. When you're stuck on something, try adding a new block of code below what you're working on and using it to debug your code. If you are new to Jupyter Notebooks, see [here](https://www.youtube.com/watch?v=inN8seMm7UI&ab_channel=TensorFlow) for a brief introduction video. If you are using Google Colab (which we recommend doing), please ensure you have changed the runtime type to use a GPU, as it will make your code run much faster.

# Imports

Run the following blocks of code to install and import and the necessary python packages.

In [1]:
!pip install pykeen

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 630 kB 35.3 MB/s 
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
    Preparing wheel metadata ... done
     |████████████████████████████████| 348 kB 60.4 MB/s 
     |████████████████████████████████| 81 kB 9.9 MB/s 
     |████████████████████████████████| 209 kB 75.5 MB/s 
     |████████████████████████████████| 78 kB 8.3 MB/s 
     |████████████████████████████████| 256 kB 68.5 MB/s 
     |████████████████████████████████| 50 kB 7.8 MB/s 
     |████████████████████████████████| 112 kB 62.4 MB/s 
     |████████████████████████████████| 147 kB 63.7 MB/s 
  Created wheel for click-default-group: filename=click_default_group-1.2.2-py3-none-any.whl size=3385 sha256=8f69b4c7edb6f70b04db405a2ca294cb8941550bdc76fad4c5afb7ba4f09ea8f
  Stored in directory: /root/.cache/pip/w

In [2]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pykeen
import argparse
import json
import os
import random

from torch.utils.data import Dataset, DataLoader
from torch.utils import data as torch_data
from sklearn.metrics import average_precision_score
from pykeen.datasets import Nations

# Dataset

## Loading Nations from `pykeen`
We will use `pykeen` to load the `Nations` dataset, which is a small knowledge graph with 14 entities, 55 relations, and 1992 triples describing countries and their political relationships.

We first define a function to convert the `pykeen` datasets to lists of triples. We then create 3 lists of triples: one for training, one for validation, and one for testing.

In [3]:
def create_triples_from_pykeen_dataset(dataset: pykeen.triples.triples_factory.TriplesFactory):
    slcwa_instances=dataset.create_slcwa_instances(
        batch_size=1,
        shuffle=True,
    )
    positive_dataset = [tuple(batch.positives[0].tolist()) for batch in slcwa_instances]
    return positive_dataset

In [4]:
dataset = Nations()
train_triples = create_triples_from_pykeen_dataset(dataset.training)
valid_triples = create_triples_from_pykeen_dataset(dataset.validation)
test_triples = create_triples_from_pykeen_dataset(dataset.testing)

In [5]:
dataset.summarize()

Nations (create_inverse_triples=False)
Name        Entities    Relations      Triples
----------  ----------  -----------  ---------
Training    14          55                1592
Testing     14          55                 201
Validation  14          55                 199
Total       -           -                 1992
Head    Relation            tail
------  ------------------  ------
brazil  blockpositionindex  china
brazil  blockpositionindex  cuba
brazil  blockpositionindex  poland
brazil  blockpositionindex  ussr
brazil  booktranslations    uk



#### Task 1
Define a function `id_triple_as_labels` that takes in a triple of entity/relation ids and returns a triple of labels.

*Hint: use the dictionaries `id2entity` and `id2relation` provided below*

In [6]:
id2entity = dataset.training.entity_id_to_label
id2relation = dataset.training.relation_id_to_label

### SOLUTION 
def id_triple_as_labels(triple: tuple):
    return (id2entity[triple[0]],
            id2relation[triple[1]],
            id2entity[triple[2]])


We can then run the following cell to see some of the facts from the **Nations** dataset.

In [7]:
for i in range(10):
    print(train_triples[i])
    print(id_triple_as_labels(train_triples[i]))

(5, 27, 11)
('india', 'ngo', 'uk')
(12, 3, 8)
('usa', 'blockpositionindex', 'jordan')
(1, 7, 3)
('burma', 'commonbloc1', 'cuba')
(2, 5, 13)
('china', 'boycottembargo', 'ussr')
(12, 9, 9)
('usa', 'conferences', 'netherlands')
(3, 38, 11)
('cuba', 'relintergovorgs', 'uk')
(0, 45, 11)
('brazil', 'timesinceally', 'uk')
(3, 9, 13)
('cuba', 'conferences', 'ussr')
(11, 30, 10)
('uk', 'officialvisits', 'poland')
(10, 19, 4)
('poland', 'independence', 'egypt')


## Negative Sampling
Simply training on positive facts will not suffice, since the model will then learn to just maximise the similarity measure for every possible fact in the database. Thus, for each positive fact, we need to sample a set of corrupted facts.

#### Task 2
First, define a function `get_corrupted_entities` that takes in a single positive sample and returns an array of corrupted entities. The positive sample is a tuple of integers, representing the IDs of the entities/relation.

The function should also take in `train_dataset`, as you will need to access `train_dataset.negative_sample_size` (the number of corrupted entities to sample), `train_dataset.nentity` (the number of entities in the knowledge graph), and possibly `train_dataset.true_head` / `train_dataset.true_tail` (dictionaries providing the set of all known true triples for the given head / tail). The definition of `TrainDataset` can be found further below if you need to refer to it.

The function should also take in `mode`, which denotes corrupting the head entity if `mode == 'head'` and likewise the tail entity if `mode == 'tail'`. Your output should be a numpy array with shape `[train_dataset.negative_sample_size]`.

Note: as an optional extra, you must sample corrupted entities such that the resulting triples are not known to be true in the knowledge graph. If you get stuck on this, rather implement a simpler solution first and then come back to it.

In [8]:
### SOLUTION 
def get_corrupted_entities(positive_sample: tuple, 
                           train_dataset: Dataset, 
                           mode: str) \
                           -> np.ndarray:

    head, relation, tail = positive_sample
    
    num_samples = train_dataset.negative_sample_size

    # List of all entity ids
    entity_ids = np.arange(train_dataset.nentity)

    if mode == 'head':
        all_partners = np.delete(entity_ids, np.argwhere(entity_ids == head))
        true_partners = train_dataset.true_head[(relation, tail)]
    else:
        all_partners = np.delete(entity_ids, np.argwhere(entity_ids == tail))
        true_partners = train_dataset.true_tail[(head, relation)]

    # entities not known to be in a relation with our given entity
    false_partners = np.setdiff1d(all_partners, true_partners)

    # sample num_samples negative samples uniformly at random
    negative_sample = np.random.choice(false_partners, 
                                       size=num_samples,
                                       replace=True) # maybe False?

    return negative_sample


#### Task 3
Next, we can create the negative samples using the corrupted entities. Define a function `get_negative_sample` which takes in a positive sample and corrupted head/tail entities, which will be created by the `get_corrupted_entities` function we defined above (do not call to the above function, the corrupted entities will be passed in as arguments).

The argument `positive_sample` is a tuple `(head, relation, tail)`. The arguments `corrupted_head_entities` and `corrupted_tail_entities` should have shape `[negative_sample_size]` each.

Your function should return a `numpy` array with shape `[2* negative_sample_size, 3]`, by combining the corrupted entities with the positive sample.

In [9]:
### SOLUTION 
def get_negative_sample(positive_sample: tuple, 
                        corrupted_head_entities: np.ndarray, 
                        corrupted_tail_entities: np.ndarray) \
                        -> np.ndarray:
    corrupted_head_samples = np.zeros((corrupted_head_entities.shape[0], 3))
    corrupted_head_samples[:, 0] = corrupted_head_entities
    corrupted_head_samples[:, 1:3] = positive_sample[1:3]

    corrupted_tail_samples = np.zeros((corrupted_tail_entities.shape[0], 3))
    corrupted_head_samples[:, 2] = corrupted_tail_entities
    corrupted_head_samples[:, 0:2] = positive_sample[0:2]
    
    return np.concatenate((corrupted_head_samples, corrupted_tail_samples), axis=0)


## Train Dataset
Now that we have written the functions we need to perform our negative sampling, let's combine everything together to create our `torch` training dataset.

Notice that in the `__get_item__` function, we convert the samples to `torch` tensors before we return them. Up until this point, we have been working with `numpy` arrays; `torch` tensors have the same structure, but are optimised to run on the `GPU` and can also track gradients to be used for optimising parameters.

In [10]:
class TrainDataset(Dataset):
    def __init__(self, triples, nentity, nrelation, negative_sample_size):
        self.len = len(triples)
        self.triples = triples # all training triples
        self.triple_set = set(triples) # unique triples
        self.nentity = nentity # number of entities in the knowledge graph
        self.nrelation = nrelation # number of relations in the knowledge graph
        self.negative_sample_size = negative_sample_size // 2 # Half from heads, half from tails

        # known triples for the given heads / tails
        self.true_head, self.true_tail = self.get_true_head_and_tail(self.triples)
        
    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):
        '''
        Get an item from the Dataset.
        '''
        # Fetch a positive sample
        positive_sample = self.triples[idx]

        # Sample corrupted head and tail entities
        corrupted_head_entities = get_corrupted_entities(positive_sample, self, 'head')
        corrupted_tail_entities = get_corrupted_entities(positive_sample, self, 'tail')

        # Create the negative sample
        negative_sample = get_negative_sample(positive_sample, corrupted_head_entities, corrupted_tail_entities)

        # Convert samples to torch tensors
        negative_sample = torch.LongTensor(negative_sample)
        positive_sample = torch.LongTensor(positive_sample)
            
        return positive_sample, negative_sample
    
    @staticmethod
    def collate_fn(data):
        positive_sample = torch.stack([_[0] for _ in data], dim=0)
        negative_sample = torch.stack([_[1] for _ in data], dim=0)
        return positive_sample, negative_sample
    
    @staticmethod
    def get_true_head_and_tail(triples):
        '''
        Build a dictionary of true triples that will
        be used to filter these true triples for negative sampling
        '''
        
        true_head = {}
        true_tail = {}

        for head, relation, tail in triples:
            if (head, relation) not in true_tail:
                true_tail[(head, relation)] = []
            true_tail[(head, relation)].append(tail)
            if (relation, tail) not in true_head:
                true_head[(relation, tail)] = []
            true_head[(relation, tail)].append(head)

        for relation, tail in true_head:
            true_head[(relation, tail)] = np.array(list(set(true_head[(relation, tail)])))
        for head, relation in true_tail:
            true_tail[(head, relation)] = np.array(list(set(true_tail[(head, relation)])))                 

        return true_head, true_tail


## Test Dataset
We will use a seperate dataset for our testing, with the batch size always set to 1. The test dataset will have two modes: `'head-batch'` and `'tail-batch'`. In the first mode, the dataset should return a positive sample and a list of all possible heads, and similarly for the second mode.

Since we are doing filtered evaluation, we do not want triples which are known to be true to affect the ranking. Thus, we will filter the heads / tails out of our samples that yield triples which are known to be true.

#### Task 4
Write a function `get_filtered_test_sample` that takes in a positive triple (`head, relation, tail`) and the `test_dataset` (which can be found further below). The function should return a list of size `test_dataset.nentity`, where each element is the ID of an entity. It is important for the downstream ranking that we ensure the list is this size.

To filter the sample, if for some entity `x`, `(x, relation, tail)` already appears in the set of known triples, instead replace it with `head`. The set of known triples can be accessed by `test_dataset.triple_set`. You will need to check the value of `test_dataset.mode` and return a list of entities accordingly.

In [11]:
### SOLUTION 
def get_filtered_test_sample(head, relation, tail, 
                             test_dataset: Dataset) \
                             -> list:
    # triples known to be true
    known_relations = test_dataset.triple_set

    # sampled test triples
    test_relations = np.asarray(test_dataset.triples)

    if test_dataset.mode == 'head-batch':

        # filtering

        # condition[x] = true if (x, relation, tail) in known_relations
        
        # is there an easier way to check which rows in an array 
        # appear in a list?
        condition = (test_relations[:, None] == known_relations).all(axis=2).any(axis=0) & \
                    (test_relations[:, 1] == relation) & \
                    (test_relations[:, 2] == tail)

        filtered_entities = np.where(condition, head, test_relations[:, 0])

    elif test_dataset.mode == 'tail-batch':

        # condition[x] = true if (head, relation, x) in known_relations
        condition = (test_relations[:, None] == known_relations).all(axis=2).any(axis=0) & \
                    (test_relations[:, 0] == head) & \
                    (test_relations[:, 1] == relation)
        filtered_entities = np.where(condition, tail, test_relations[:, 2])

    else:
        raise ValueError('negative batch mode %s not supported' % test_dataset.mode)

    # sample num_samples filtered test samples uniformly at random
    filtered_sample = np.random.choice(filtered_entities, 
                                       size=test_dataset.nentity,
                                       replace=True) # maybe False?
         
    return filtered_sample


Now we can use our filtered test sampling function to define our test dataset.

In [12]:
class TestDataset(Dataset):
    def __init__(self, triples, all_true_triples, nentity, nrelation, mode):
        self.len = len(triples)
        self.triple_set = set(all_true_triples) # set of all known true triples
        self.triples = triples # test triples
        self.nentity = nentity
        self.nrelation = nrelation
        self.mode = mode # 'head-batch' or 'tail-batch'

    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):
        head, relation, tail = self.triples[idx] # fetch a positive sample from the test triples

        # get the filtered sample using the function we defined
        filtered_sample = get_filtered_test_sample(head, relation, tail, self)

        # convert to torch tensors
        filtered_sample = torch.LongTensor(filtered_sample)
        positive_sample = torch.LongTensor((head, relation, tail))
            
        return positive_sample, filtered_sample, self.mode
    
    @staticmethod
    def collate_fn(data):
        positive_sample = torch.stack([_[0] for _ in data], dim=0)
        negative_sample = torch.stack([_[1] for _ in data], dim=0)
        mode = data[0][2]
        return positive_sample, negative_sample, mode

## Dataset Iterator
As a final step towards constructing our datasets, we define a class that allows us to convert `torch` dataloaders into python iterators, which will make it simpler for us to define training and testing step functions.

In [13]:
class OneShotIterator(object):
    def __init__(self, dataloader):
        self.iterator = self.one_shot_iterator(dataloader)
        
    def __next__(self):
        return next(self.iterator)
    
    @staticmethod
    def one_shot_iterator(dataloader):
        '''
        Transform a PyTorch Dataloader into python iterator
        '''
        while True:
            for data in dataloader:
                yield data

We will create the actual train and test datasets in code further below, but here follows some code for constructing them, in case you would like to use it for debugging.

In [14]:
# make the train dataloader
train_dataloader = DataLoader(
    TrainDataset(train_triples,
                 14, # nentity
                 55, # nrelation
                 128), # negative sampling size
    batch_size=500,
    shuffle=True, 
    num_workers=1,
    collate_fn=TrainDataset.collate_fn
)
# convert to an iterator
train_iterator = OneShotIterator(train_dataloader)
# get a sample from the train iterator
positive_sample, negative_sample = next(train_iterator)

If your code is correct, the following output should be `torch.Size([500, 128, 3])`.

In [15]:
negative_sample.size()

torch.Size([500, 128, 3])

In [16]:
# make the test dataloader
known_true_triples = train_triples + valid_triples + test_triples
test_dataloader_head = DataLoader(
    TestDataset(
        test_triples,
        known_true_triples,
        14, # nentity
        55, # nrelation
        'head-batch'
    ), 
    batch_size=1,
    num_workers=1, 
    collate_fn=TestDataset.collate_fn
)

If your code is correct, the following output should be `torch.Size([1, 14])`.

In [17]:
for positive_sample, negative_sample, mode in test_dataloader_head:
    print(negative_sample.size())
    break

torch.Size([1, 14])


# Model
We now define our model, `KGEModel`. It is built in such a way that we can implement different dissimilarity measures within it.

From here onwards, we will be working with `torch` tensors instead of `numpy` arrays, so make sure you are using `torch` operations.

## Parameter Initialisation
We will use `torch.nn.Parameter` to store our embeddings for entities and relations. We define a function `init_params` which initialises an embedding tensor of the given size and randomly samples values from the uniform distribution `[-embedding_range, embedding_range]`.

In [18]:
def init_params(tensor_size: tuple, embedding_range: float) -> nn.Parameter:
    embedding = nn.Parameter(torch.zeros(tensor_size))
    nn.init.uniform_(
        tensor=embedding, 
        a=-embedding_range, 
        b=embedding_range
    )
    return embedding

## Scoring Function
Different KG embedding models use different dissimilarity measures (aka scoring functions). We will define one for **TransE** and optionally define one for **RotatE**.

#### Task 5
Define a scoring function for **TransE** that takes in the head, relation, and tail, and returns a score for the triple. Each argument tensor has size `[batch_size, embedding_size]`. You may use either the $L_1$ or $L_2$ norm. Your output tensor should have size `[batch_size]`.

In [45]:
### BEGIN SOLUTION
def TransE(head, relation, tail):
    # using L2 norm to score distance
    return torch.norm(head + relation - tail, p=2, dim=1)

### END SOLUTION

### Task 6 (Optional)
This is an optional task. You should get **TransE** working completely first and then come back to this.

Define a scoring function for **RotatE**. The head and tail will have size `[batch_size, 2 * embedding_size]` to store both the real and imaginary parts of the entities. *Hint: you can use `torch.chunk()` to split the tensor into its real and imaginary components*.

The relation will have size `[batch_size, embedding_size]`, representing the phase $\theta$ of the relation. *Hint: The real and imaginary components of the relation can be computed with `torch.cos` and `torch.sin`*.

In [47]:
### BEGIN SOLUTION
def RotatE(head, relation, tail):
    def rot(u, r):
        a, b = torch.chunk(u, 2, dim=1)
        c = torch.cos(r)
        d = torch.sin(r)
        # u = a + i*b
        # e^(i*r) = c + i*d

        # multiplying => a*c - b*d + i*(a*d + b*c)
        return torch.cat([a*c - b*d, a*d + b*c], dim=1)

    # using L2 norm to score distance
    return torch.norm(rot(head, relation) - tail, p=2, dim=1)
### END SOLUTION

## Full Model Definition
Now we can use our scoring function to define the model. Notice that in the `forward` function, we use `torch.index_select` to fetch the entity / relation embeddings from the their indices. `sample` is a tensor with the size `[batch_size, 3]`.

In [21]:
class KGEModel(nn.Module):
    def __init__(self, model_name, nentity, nrelation, hidden_dim, gamma, 
                 double_entity_embedding=False, double_relation_embedding=False):
        super(KGEModel, self).__init__()
        self.model_name = model_name
        self.nentity = nentity
        self.nrelation = nrelation
        self.hidden_dim = hidden_dim
        self.epsilon = 2.0
        
        self.gamma = nn.Parameter(              # error margin hyperparameter
            torch.Tensor([gamma]), 
            requires_grad=False
        )
        
        self.embedding_range = nn.Parameter(
            torch.Tensor([(self.gamma.item() + self.epsilon) / hidden_dim]), 
            requires_grad=False
        )
        
        self.entity_dim = hidden_dim*2 if double_entity_embedding \
                     else hidden_dim
        self.relation_dim = hidden_dim*2 if double_relation_embedding \
                       else hidden_dim
        
        # Create entity and relation embeddings
        self.entity_embedding = init_params(tensor_size=(nentity, self.entity_dim),
                                             embedding_range=self.embedding_range.item())
        self.relation_embedding = init_params(tensor_size=(nrelation, self.relation_dim),
                                              embedding_range=self.embedding_range.item())
        
        # This code supports easily adding new models like RotatE and ComplEx
        # Do not forget to modify this line when you add a new model in the
        # "forward" function
        if model_name not in ['TransE', 'RotatE']:
            raise ValueError('model %s not supported' % model_name)
        
        if model_name == 'RotatE' and not double_entity_embedding:
            raise ValueError('RotatE should use --double_entity_embedding')
        if model_name == 'TransE' and double_entity_embedding:
            raise ValueError('TransE should not use --double_entity_embedding') 
        
    def forward(self, sample):
        '''
        Forward function that calculate the score of a batch of triples.
        Sample is a batch of triples.
        '''
        head = torch.index_select(
            self.entity_embedding, 
            dim=0, 
            index=sample[:,0]
        )
        
        relation = torch.index_select(
            self.relation_embedding, 
            dim=0, 
            index=sample[:,1]
        )
        
        tail = torch.index_select(
            self.entity_embedding, 
            dim=0, 
            index=sample[:,2]
        )
        
        # Other models can be added here
        dissimilarity_measure = {
            'TransE': TransE,
            'RotatE': RotatE,
        }
        
        if self.model_name in dissimilarity_measure:
            score = dissimilarity_measure[self.model_name](head, relation, tail)
        else:
            raise ValueError('model %s not supported' % self.model_name)
        
        return score

# Training
In this section, we will first write a function to compute the model loss given a set of positive and negative samples, and then use it to define a single training step for the model.

#### Task 7
Before we can define a training step for our model, we must define a loss function for the model. We use negative sampling loss (from the **RotatE** paper), but without the self-adversarial parameter. Remember to refer to the lecture slides if you get stuck on this.

Define a function `get_model_loss` which takes in the `KGEModel`, the positive sample, and the negative sample, and returns a tuple of the loss, the positive sample loss, and the negative sample loss.

The positive sample will have size `[batch_size, 3]`, the negative sample will have size `[batch_size * negative_sampling_size, 3]`, and the margin $\gamma$ can be accessed through `model.gamma`.

In [22]:
### BEGIN SOLUTION
def get_model_loss(model: KGEModel, 
                   positive_sample: torch.tensor, 
                   negative_sample: torch.tensor) -> tuple:
    batch_size = positive_sample.size(dim=0)
    sig = torch.sigmoid
    log = torch.log
    gamma = model.gamma
    
    # using the negative log loss from the RotatE paper
    positive_sample_loss = torch.mean(-log(sig(gamma - model(positive_sample))))

    negative_score = torch.reshape(model(negative_sample), [batch_size, -1])
    # negative_score.shape = [batch_size, negative_sampling_size]

    # for each data point in the batch we want to compute the mean
    # negative log loss over the corresponding negative samples
    # the overall negative sample loss is the mean over the means 

    # since the average of averages is just the average over all 
    # elements in the tensor, we can get away with only computing
    # the mean once
    negative_sample_loss = -torch.mean(log(sig(negative_score - gamma)))

    loss = positive_sample_loss + negative_sample_loss

    return loss, positive_sample_loss, negative_sample_loss
### END SOLUTION

We can now define a single train step for the model.

In [23]:
def train_step(model, optimizer, train_iterator, args):
    '''
    A single train step. Apply back-propation and return the loss
    '''

    model.train() # tell the torch model it's about to be trained

    optimizer.zero_grad() # explicitly set gradients to 0 before starting backprop

    positive_sample, negative_sample = next(train_iterator) # fetch samples from the dataset

    # reshape the negative sample
    # it will now have shape [batch_size * negative_sampling_size, 3]
    negative_sample = torch.reshape(negative_sample, (-1, 3))

    # move tensors to GPU
    if args.cuda:
        positive_sample = positive_sample.cuda()
        negative_sample = negative_sample.cuda()

    # compute the loss
    loss, positive_sample_loss, negative_sample_loss = get_model_loss(model, positive_sample, negative_sample)

    # apply loss
    loss.backward()
    optimizer.step()

    log = {
        'positive_sample_loss': positive_sample_loss.item(),
        'negative_sample_loss': negative_sample_loss.item(),
        'loss': loss.item()
    }

    return log

# Testing
In this section, we will first write a function to get the ranking of a positive entity compared to its corrupted counterparts, and then use that to define a single test step for the model.

#### Task 8

Define a function `get_ranking` which takes in entity scores and the index of the positive entity. The function should return an integer representing the rank of the score of the positive entity in relation to the other entities. Note that a rank of 1 represents having the *lowest* score.

`entity_scores` has size `[nentity]`. Recall from the function we defined further above that we filtered out known true triples by replacing the corrupted heads with the actual head, so some of the entity scores may actually be that of the positive entity, even when they are not in the index of that entity.

*Hint: torch.argsort will be very useful for this task*.

In [32]:
### BEGIN SOLUTION
def get_ranking(entity_scores: torch.tensor, positive_entity: int) -> int:
    positive_score = entity_scores[positive_entity]


    duplicate_removal_mask = torch.logical_not(torch.isclose(entity_scores, positive_score))
    duplicate_removal_mask[positive_entity] = True

    entity_scores = entity_scores[duplicate_removal_mask]
    # print(entity_scores.shape)
    sorted_scores, _ = torch.sort(entity_scores)

    # get the first index of scores equal to the positive score
    # for some reason this gives a high MR value still :(
    ranking = torch.isclose(sorted_scores, positive_score).type(torch.uint8).nonzero().item()

    return (ranking+1)
### END SOLUTION

We can now define a single test step for the model, using the ranking function to compute MRR, MR, and HITS@k metrics.

In [25]:
def test_step(model, test_triples, all_true_triples, args):
    '''
    Evaluate the model on test or valid datasets
    '''
    
    model.eval()
    
    #Prepare dataloader for evaluation
    test_dataloader_head = DataLoader(
        TestDataset(
            test_triples, 
            all_true_triples, 
            args.nentity, 
            args.nrelation, 
            'head-batch'
        ), 
        batch_size=args.test_batch_size,
        num_workers=max(1, args.cpu_num//2), 
        collate_fn=TestDataset.collate_fn
    )

    test_dataloader_tail = DataLoader(
        TestDataset(
            test_triples, 
            all_true_triples, 
            args.nentity, 
            args.nrelation, 
            'tail-batch'
        ), 
        batch_size=args.test_batch_size,
        num_workers=max(1, args.cpu_num//2), 
        collate_fn=TestDataset.collate_fn
    )
    
    test_dataset_list = [test_dataloader_head, test_dataloader_tail]
    
    logs = []

    step = 0
    total_steps = sum([len(dataset) for dataset in test_dataset_list])

    # torch.no_grad() since we don't need to track gradients when we're testing
    with torch.no_grad():
        for test_dataset in test_dataset_list: # each of head / tail
            for positive_sample, negative_sample, mode in test_dataset:
                # take a sample from the test dataset
                if args.cuda:
                    positive_sample = positive_sample.cuda()
                    negative_sample = negative_sample.cuda()

                batch_size = positive_sample.size(0)
                assert batch_size == 1, 'evaluation batch size must be set to 1'
                
                # build the negative sample from the entities
                # currently, negative_sample is just a list of entities
                # [1, 14, 3] for Nations
                built_negative_sample = torch.zeros((batch_size, negative_sample.size()[1], 3), dtype=int)
                if args.cuda:
                    built_negative_sample = built_negative_sample.cuda()
                
                if mode == 'head-batch':
                    built_negative_sample[:, :, 0] = negative_sample
                    built_negative_sample[:, :, 1:3] = positive_sample[:, 1:3].unsqueeze(dim=1).expand((-1, built_negative_sample.size(1), -1))
                else:
                    built_negative_sample[:, :, 2] = negative_sample
                    built_negative_sample[:, :, 0:2] = positive_sample[:, 0:2].unsqueeze(dim=1).expand((-1, built_negative_sample.size(1), -1))
                
                # get the scores for each entity
                negative_sample = built_negative_sample.reshape((-1, 3))
                entity_scores = model(negative_sample)

                # retrieve the positive entity
                if mode == 'head-batch':
                    positive_entity = positive_sample[:, 0].item()
                elif mode == 'tail-batch':
                    positive_entity = positive_sample[:, 2].item()
                else:
                    raise ValueError('mode %s not supported' % mode)

                # get the ranking of the positive entity
                ranking = get_ranking(entity_scores, positive_entity)
                
                # compute and append logs
                logs.append({
                    'MRR': 1.0/ranking,
                    'MR': float(ranking),
                    'HITS@1': 1.0 if ranking <= 1 else 0.0,
                    'HITS@3': 1.0 if ranking <= 3 else 0.0,
                    'HITS@10': 1.0 if ranking <= 10 else 0.0,
                })

                if step % args.test_log_steps == 0:
                    print('Evaluating the model... (%d/%d)' % (step, total_steps))

                step += 1

    metrics = {}
    for metric in logs[0].keys():
        metrics[metric] = sum([log[metric] for log in logs])/len(logs)

    return metrics

# Running
To make the running of experiments easier, we will define several help functions.

## Arguments
`argparse` is a very useful library for managing program arguments, particulary when executing from the command line. Default arguments are defined here. If you want to change argument values, do not do it in this block of code, rather change them in the arguments that are passed through to the program (see further below).

In [26]:
def parse_args(args=None):
    parser = argparse.ArgumentParser(
        description='Training and Testing Knowledge Graph Embedding Models',
        usage='train.py [<args>] [-h | --help]'
    )

    parser.add_argument('--cuda', action='store_true', help='use GPU')
    
    parser.add_argument('--do_train', action='store_true')
    parser.add_argument('--do_valid', action='store_true')
    parser.add_argument('--do_test', action='store_true')
    parser.add_argument('--evaluate_train', action='store_true', help='Evaluate on training data')
    
    parser.add_argument('--model', default='TransE', type=str)
    parser.add_argument('-de', '--double_entity_embedding', action='store_true')
    parser.add_argument('-dr', '--double_relation_embedding', action='store_true')
    
    parser.add_argument('-n', '--negative_sample_size', default=128, type=int)
    parser.add_argument('-d', '--hidden_dim', default=100, type=int, help='Embedding size')
    parser.add_argument('-g', '--gamma', default=2.0, type=float, help='Fixed margin parameter')
    parser.add_argument('-b', '--batch_size', default=1024, type=int)
    parser.add_argument('--test_batch_size', default=1, type=int, help='valid/test batch size (must be 1)')
    
    parser.add_argument('-lr', '--learning_rate', default=0.0001, type=float)
    parser.add_argument('-cpu', '--cpu_num', default=1, type=int)
    parser.add_argument('--max_steps', default=100, type=int)
    
    parser.add_argument('--valid_steps', default=20, type=int, help='how often to check accuracy on validation dataset')
    parser.add_argument('--log_steps', default=10, type=int, help='train log every xx steps')
    parser.add_argument('--test_log_steps', default=1000, type=int, help='valid/test log every xx steps')
    
    parser.add_argument('--nentity', type=int, default=0, help='DO NOT MANUALLY SET')
    parser.add_argument('--nrelation', type=int, default=0, help='DO NOT MANUALLY SET')
    
    parser.parse_args(args)
    return parser.parse_args(args)

## Logging
The below function is used to help with logging metrics.

In [27]:
def log_metrics(mode, step, metrics):
    '''
    Print the evaluation logs
    '''
    for metric in metrics:
        print('%s %s at step %d: %f' % (mode, metric, step, metrics[metric]))

## Main Program Loop
We can finally bring everything we've done together into the main program loop. Please refer to the comments in the code to understand how it operates.

In [40]:
def main(args):
    if (not args.do_train) and (not args.do_valid) and (not args.do_test):
        raise ValueError('one of train/val/test mode must be chosen.')
    
    # use CUDA if possible
    args.cuda = torch.cuda.is_available()
    
    # print dataset parameters
    entity2id = dataset.training.entity_to_id
    relation2id = dataset.training.relation_to_id

    nentity = len(entity2id)
    nrelation = len(relation2id)
    
    args.nentity = nentity
    args.nrelation = nrelation
    
    print('Model: %s' % args.model)
    print('#entity: %d' % nentity)
    print('#relation: %d' % nrelation)
    
    print('#train: %d' % len(train_triples))
    print('#valid: %d' % len(valid_triples))
    print('#test: %d' % len(test_triples))
    
    # all true triples
    all_true_triples = train_triples + valid_triples + test_triples
    
    # create the model
    kge_model = KGEModel(
        model_name=args.model,
        nentity=nentity,
        nrelation=nrelation,
        hidden_dim=args.hidden_dim,
        gamma=args.gamma,
        double_entity_embedding=args.double_entity_embedding,
        double_relation_embedding=args.double_relation_embedding
    )
    
    # output the model params
    print('\nModel Parameter Configuration:')
    for name, param in kge_model.named_parameters():
        print('Parameter %s: %s, require_grad = %s' % (name, str(param.size()), str(param.requires_grad)))

    if args.cuda:
        kge_model = kge_model.cuda()
    
    if args.do_train:
        # set training dataloader iterator
        train_dataloader = DataLoader(
            TrainDataset(train_triples, nentity, nrelation, args.negative_sample_size), 
            batch_size=args.batch_size,
            shuffle=True, 
            num_workers=max(1, args.cpu_num//2),
            collate_fn=TrainDataset.collate_fn
        )
        train_iterator = OneShotIterator(train_dataloader)

        # set training configuration
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, kge_model.parameters()), 
            lr=args.learning_rate
        )

    init_step = 1
    step = init_step

    # do an initial evaluation to see metrics for a random model
    print('\nEvaluating initial model on Valid Dataset...')
    metrics = test_step(kge_model, valid_triples, all_true_triples, args)
    log_metrics('Valid', step, metrics)
    
    # log training parameters
    print('\nStart Training...')
    print('init_step = %d' % init_step)
    print('batch_size = %d' % args.batch_size)
    print('hidden_dim = %d' % args.hidden_dim)
    print('gamma = %f' % args.gamma)
        
    if args.do_train:
        print('learning_rate = %f' % args.learning_rate)

        training_logs = []
        
        # training loop
        for step in range(init_step, args.max_steps):
            # perform a single train step
            log = train_step(kge_model, optimizer, train_iterator, args)
            
            # record the logs
            training_logs.append(log)
                
            # check if logs should be displayed
            if step % args.log_steps == 0:
                metrics = {}
                for metric in training_logs[0].keys():
                    metrics[metric] = sum([log[metric] for log in training_logs])/len(training_logs)
                print('\nTraining metrics...')
                log_metrics('Training average', step, metrics)
                training_logs = []
                
            # check if metrics should be reported on the validation set
            if args.do_valid and step % args.valid_steps == 0:
                print('\nEvaluating on Valid Dataset...')
                metrics = test_step(kge_model, valid_triples, all_true_triples, args)
                log_metrics('Valid', step, metrics)
        
    # compute final metrics after training is complete

    if args.do_valid:
        print('Evaluating on Valid Dataset...')
        metrics = test_step(kge_model, valid_triples, all_true_triples, args)
        log_metrics('Valid', step, metrics)
    
    if args.do_test:
        print('Evaluating on Test Dataset...')
        metrics = test_step(kge_model, test_triples, all_true_triples, args)
        log_metrics('Test', step, metrics)
    
    if args.evaluate_train:
        print('Evaluating on Training Dataset...')
        metrics = test_step(kge_model, train_triples, all_true_triples, args)
        log_metrics('Test', step, metrics)

The below code can be used to run the main program loop. Model arguments can be adjusted by changing / adding / removing the arguments.

In [48]:
if __name__ == '__main__':
    main(parse_args(['--do_train', '--do_valid', '--do_test',
                     '--model', 'TransE',
                     #'--double_entity_embedding',
                     '--max_steps', '1000', '--valid_steps', '20', '--log_steps', '10']))

Model: TransE
#entity: 14
#relation: 55
#train: 1592
#valid: 199
#test: 201

Model Parameter Configuration:
Parameter gamma: torch.Size([1]), require_grad = False
Parameter embedding_range: torch.Size([1]), require_grad = False
Parameter entity_embedding: torch.Size([14, 100]), require_grad = True
Parameter relation_embedding: torch.Size([55, 100]), require_grad = True

Evaluating initial model on Valid Dataset...
Evaluating the model... (0/398)
Valid MRR at step 1: 0.291607
Valid MR at step 1: 6.879397
Valid HITS@1 at step 1: 0.145729
Valid HITS@3 at step 1: 0.253769
Valid HITS@10 at step 1: 0.771357

Start Training...
init_step = 1
batch_size = 1024
hidden_dim = 100
gamma = 2.000000
learning_rate = 0.000100

Training metrics...
Training average positive_sample_loss at step 10: 0.185437
Training average negative_sample_loss at step 10: 1.853937
Training average loss at step 10: 2.039374

Training metrics...
Training average positive_sample_loss at step 20: 0.187705
Training average ne

KeyboardInterrupt: ignored